<a href="https://colab.research.google.com/github/nikitask14/neural-networks-pytorch-from-first-principles/blob/main/04_debugging_and_generalisation/14_dropout_weightdecay_and_earlystopping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sitting 14 — Dropout, Weight Decay, and Early Stopping

## Objective

In Sitting 13, we learned to **detect overfitting** from learning curves:

- training loss continues to decrease,
- validation loss stops improving,
- validation loss may begin to rise.

In this sitting, we study three ways to respond to overfitting:

1. **Dropout** - changes hidden activations during training.
2. **Weight decay** — discourages overly large parameter values during updates.
3. **Early stopping** — stops training when validation performance stops improving.

We keep the experiments separate so we can understand what each method is doing.

## Important experimental rule

For every new experiment we recreate:

- the model,
- the optimiser,
- the training-loss list,
- the validation-loss list,
- and any early-stopping variables.

This ensures that one experiment does not continue from the state of another.

# Experiment 1 — Early stopping only

## Why?

In Sitting 13, validation loss eventually began to rise even though training loss continued to decrease.

Early stopping watches validation loss and stops training after a chosen number of consecutive epochs without improvement.

Here we use 'patience = 100' so small short-term fluctuations do not stop training immediately.


In [ ]:
# Fresh experiment state
model = MyModel()

optimiser = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

loss_fn = nn.BCEWithLogitsLoss()

train_losses = []
val_losses = []

best_val_loss = float("inf")
counter = 0
patience = 100

for epoch in range(5000):

    model.train()

    optimiser.zero_grad()

    train_logits = model(x_train)
    train_loss = loss_fn(train_logits, y_train)

    train_loss.backward()
    optimiser.step()

    model.eval()

    with torch.no_grad():

        # Recalculate training loss using the UPDATED model
        train_logits_eval = model(x_train)
        train_loss_eval = loss_fn(train_logits_eval, y_train)

        # Validation loss using the SAME updated model
        val_logits = model(x_val)
        val_loss = loss_fn(val_logits, y_val)

    train_losses.append(train_loss_eval.item())
    val_losses.append(val_loss.item())

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

    if epoch % 50 == 0:
        print(
            f"Epoch: {epoch}, "
            f"Training Loss: {train_loss_eval.item():.6f}, "
            f"Validation Loss: {val_loss.item():.6f}"
        )

print("Stored training losses:", len(train_losses))
print("Stored validation losses:", len(val_losses))


In [ ]:
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


### Early-stopping mental model

- validation improves → save the new best value and reset *counter*
- validation does not improve → *counter += 1*
- *counter* reaches *patience* → stop training

Early stopping changes **how long we train**.

We are not yet saving/restoring the best model. That will come later with *state_dict()*.


# Experiment 2 — Weight decay only

## Why?

Weight decay discourages the optimiser from allowing parameter values to become unnecessarily large.

It does **not** guarantee that overfitting disappears.

For this experiment:

- no dropout,
- no early stopping,
- *weight_decay=0.001*,
- 2500 epochs.


In [ ]:
# Fresh experiment state
model = MyModel()

optimiser = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=0.001
)

loss_fn = nn.BCEWithLogitsLoss()

train_losses = []
val_losses = []

for epoch in range(2500):

    model.train()

    optimiser.zero_grad()

    train_logits = model(x_train)
    train_loss = loss_fn(train_logits, y_train)

    train_loss.backward()
    optimiser.step()

    model.eval()

    with torch.no_grad():

        train_logits_eval = model(x_train)
        train_loss_eval = loss_fn(train_logits_eval, y_train)

        val_logits = model(x_val)
        val_loss = loss_fn(val_logits, y_val)

    train_losses.append(train_loss_eval.item())
    val_losses.append(val_loss.item())

    if epoch % 50 == 0:
        print(
            f"Epoch: {epoch}, "
            f"Training Loss: {train_loss_eval.item():.6f}, "
            f"Validation Loss: {val_loss.item():.6f}"
        )


In [ ]:
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


### What to look for

Weight decay may:

- delay overfitting,
- soften the rise in validation loss,
- improve the best validation loss,
- or reduce the gap between training and validation behaviour.

It does **not** mean validation loss can never rise.


# Experiment 3 — Dropout only

## Why?

Dropout randomly removes hidden activations during training.

This makes it harder for the model to rely too strongly on particular hidden pathways.

- *model.train()* → dropout active
- *model.eval()* → dropout disabled

For this experiment:

- dropout on,
- no weight decay,
- no early stopping,
- 2500 epochs.


In [ ]:
class MyModelDropout(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)

        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.layer2(x)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.layer3(x)

        return x
